Audio Grammar Scoring - SHL Labs Challenge

Author Name: Sowrya Muramshetty

 Problem Statement

The objective of this challenge is to predict grammar proficiency scores from spoken audio responses.

This task is formulated as a regression problem where the target variable is a continuous score.  

The evaluation metric used in this competition is Root Mean Squared Error (RMSE).

  Data Understanding

The dataset contains:

Audio Files:`.wav` files for train and test
  - Training audios: `/dataset/audios/train`
  - Testing audios: `/dataset/audios/test`
CsV Files:
  - `train.csv` – contains audio filenames and their grammar scores
  - `test.csv` – contains audio filenames only

Number of samples:
- Training: 409
- Testing: 197

 Preprocessing & Feature Extraction

- All audio sampled at 16 kHz
- Extracted features:
  - MFCCs (13 coefficients)
  - Mean of each coefficient across time
- Combined all features into a single vector per audio file
- Normalization applied where necessary

 Model Pipeline

- Extract features from all audio files
- Train a single XGBoost regression model
- Predict on test set
- Save predictions to submission.csv

**Evaluation Metric:** RMSE on training data (required)

Load Dataset 

import pandas as pd

import numpy as np

import os

TRAIN_CSV = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/csvs/train.csv"

TEST_CSV = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/csvs/test.csv"

TRAIN_AUDIO = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/audios/train"

TEST_AUDIO = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/audios/test"

train_df = pd.read_csv(TRAIN_CSV)

test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)

print("Test shape:", test_df.shape)

train_df.head()

Feature Extraction 

import librosa

def extract_features(file_path):

    y, sr = librosa.load(file_path, sr=16000)
    
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    
    return np.mean(mfccs.T, axis=0)

X_train = np.array([extract_features(os.path.join(TRAIN_AUDIO, f"{fname}.wav")) for fname in train_df['filename']])

y_train = train_df['label'].values

X_test = np.array([extract_features(os.path.join(TEST_AUDIO, f"{fname}.wav")) for fname in test_df['filename']])

Train Model

from xgboost import XGBRegressor

model = XGBRegressor(

    n_estimators=200, 
    
    max_depth=4, 
    
    learning_rate=0.05,
    
    random_state=42
)

model.fit(X_train, y_train)

Evaluate on Training Data

from sklearn.metrics import mean_squared_error

train_preds = model.predict(X_train)

rmse = mean_squared_error(y_train, train_preds, squared=False)

print("Training RMSE:", rmse)

Predict on Test Set

preds = model.predict(X_test)

Create Submission File

submission = pd.DataFrame({

    "filename": test_df['filename'],
    "label": preds
})

submission.to_csv("submission.csv", index=False)

print("Submission file created!")

Summary

Features: MFCC mean (13 per audio)
- Model: XGBoost Regressor
- Training RMSE: ~0.71
- Predictions saved to `submission.csv`
- Ready for Kaggle submission

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os

base_path = "/kaggle/input"

for folder in os.listdir(base_path):
    print(folder)

In [ ]:
import os
os.listdir("/kaggle/input")

In [ ]:
import os

print(os.listdir("/kaggle/input/competitions"))

In [ ]:
comp_path = "/kaggle/input/competitions/shl-audio-scoring-challenge"
print(os.listdir(comp_path))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("Current Folder:", root)
    print("Subfolders:", dirs)
    print("Files:", files)
    print("-"*60)

In [ ]:
TRAIN_CSV = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/csvs/train.csv"
TEST_CSV  = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/csvs/test.csv"

TRAIN_AUDIO_PATH = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/audios/train/"
TEST_AUDIO_PATH  = "/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/audios/test/"

In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

In [ ]:
import librosa

file_name = train_df.loc[0, "filename"] + ".wav"
file_path = TRAIN_AUDIO_PATH + file_name

audio, sr = librosa.load(file_path, sr=16000)

print("Sample Rate:", sr)
print("Duration (seconds):", len(audio)/sr)

In [ ]:
import numpy as np

def extract_features(file_path):
    audio, sr = librosa.load(file_path, sr=16000)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    return np.mean(mfcc.T, axis=0)

In [ ]:
import numpy as np
import librosa

def extract_features(file_path):
    audio, sr = librosa.load(file_path, sr=16000)

    # MFCC
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    # Delta MFCC
    delta = librosa.feature.delta(mfcc)
    delta_mean = np.mean(delta, axis=1)

    # Chroma
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)

    # Spectral Contrast
    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)
    contrast_mean = np.mean(contrast, axis=1)

    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(audio)
    zcr_mean = np.mean(zcr)

    # Combine all
    features = np.hstack([
        mfcc_mean,
        mfcc_std,
        delta_mean,
        chroma_mean,
        contrast_mean,
        zcr_mean
    ])

    return features

In [ ]:
X = []
y = []

for idx, row in train_df.iterrows():
    file_name = row["filename"] + ".wav"
    file_path = TRAIN_AUDIO_PATH + file_name
    
    features = extract_features(file_path)
    X.append(features)
    y.append(row["label"])

X = np.array(X)
y = np.array(y)

print("New feature shape:", X.shape)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

model_xgb = XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_xgb.fit(X_train, y_train)

preds = model_xgb.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, preds))
print("XGBoost RMSE:", rmse)

In [ ]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = XGBRegressor(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=3,      # reduced depth
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,    # regularization
        reg_lambda=2.0,
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    rmse_scores.append(rmse)

    print(f"Fold {fold+1} RMSE:", rmse)

print("Mean RMSE:", np.mean(rmse_scores))

In [ ]:
from xgboost import XGBRegressor

final_model = XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=42
)

final_model.fit(X, y)

In [ ]:
X_test = []

for idx, row in test_df.iterrows():
    file_name = row["filename"] + ".wav"
    file_path = TEST_AUDIO_PATH + file_name
    
    features = extract_features(file_path)
    X_test.append(features)

X_test = np.array(X_test)

In [ ]:
X_test_scaled = scaler.transform(X_test)  # scale test features same as training
test_preds = final_model.predict(X_test_scaled)

In [ ]:
submission = pd.DataFrame({
    "filename": test_df["filename"],  # filenames from test.csv
    "label": test_preds               # predicted grammar scores
})

submission.to_csv("submission.csv", index=False)

In [ ]:
import os

# List all files in the working directory
print(os.listdir("/kaggle/working/"))

In [ ]:
# Extract features for test set
X_test = np.array([extract_features(f"/kaggle/input/competitions/shl-audio-scoring-challenge/dataset/audios/test/{fn}.wav") 
                   for fn in test_df["filename"]])

# Scale test features
X_test_scaled = scaler.transform(X_test)

# Predict with final model
test_preds = final_model.predict(X_test_scaled)

# Save submission
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": test_preds
})

submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved at: {submission_path}")